# Why Domain-Specific Tokenizers Matter: General-Purpose vs. Medical Tokenization

**Teaching goal.** A general-purpose tokenizer — trained on generic web text — wastes tokens (and therefore memory, compute, and context window) on domain-specific text like biomedical abstracts. In this notebook we:

1. Compare a **general-purpose tokenizer** (`gpt2`) with a **medical tokenizer** (`microsoft/biogpt`, whose vocabulary was trained on 18M+ PubMed abstracts) on both medical and general text.
2. Discover that "medical model" does not imply "medical tokenizer": BioClinicalBERT reuses BERT's stock vocabulary, while PubMedBERT retrained its own on PubMed.
3. Fix the problem ourselves: train a medical BPE tokenizer **from scratch** with the Hugging Face `tokenizers` library and measure the improvement on held-out medical text.

**Key metric — fertility** (Rust et al., 2021, *How Good is Your Tokenizer?*): the number of tokens per word. Lower is better. A domain-matched tokenizer reaches a much lower fertility on its domain.

Run top-to-bottom in Google Colab (`Runtime → Run all`) or any Jupyter environment. Everything runs on a free Colab tier in well under 10 minutes.

In [3]:
# sacremoses is needed for the (legacy, fairseq-style) BioGPT tokenizer.
# The install only runs for packages that are missing (on Colab, usually just sacremoses).
import importlib.util
import subprocess
import sys

for pkg in ["tokenizers", "transformers", "datasets", "matplotlib", "pandas", "sacremoses"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-U", "-q", pkg], check=True)

import random

import matplotlib.pyplot as plt
import pandas as pd
import datasets
import tokenizers
import transformers

print("tokenizers  :", tokenizers.__version__)
print("transformers:", transformers.__version__)
print("datasets    :", datasets.__version__)

SEED = 42
random.seed(SEED)

tokenizers  : 0.23.2
transformers: 5.16.1
datasets    : 5.0.1


## 1. Load the two corpora

- **Medical:** real PubMed abstracts from the `scientific_papers` dataset (~133k abstracts).
- **General:** WikiText-2, a classic small benchmark of Wikipedia prose.

We carve the medical data into three disjoint slices: texts for corpus-level metrics, texts to **train** our own tokenizer, and a **held-out** slice that the training never sees (for a fair final evaluation).

In [ ]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download, list_repo_files

# `scientific_papers` is a legacy "script" dataset; recent `datasets` versions no
# longer run dataset scripts, so we load the Hub's automatic parquet export instead.
PARQUET_REV = "refs/convert/parquet"
pubmed_files = [
    hf_hub_download("scientific_papers", f, repo_type="dataset", revision=PARQUET_REV)
    for f in sorted(list_repo_files("scientific_papers", repo_type="dataset", revision=PARQUET_REV))
    if f.startswith("pubmed/train/") and f.endswith(".parquet")
]
pubmed = load_dataset("parquet", data_files=pubmed_files, split="train")

wikitext_file = hf_hub_download("wikitext", "wikitext-2-raw-v1/train-00000-of-00001.parquet",
                                  repo_type="dataset")
wikitext = load_dataset("parquet", data_files=wikitext_file, split="train")

N_METRICS = 2000   # abstracts used for corpus-level metric tables
N_TRAIN = 5000     # abstracts used to train our own tokenizer
N_HELDOUT = 1500   # abstracts never seen during training

pubmed_shuffled = pubmed.shuffle(seed=SEED)
medical_metrics_texts = pubmed_shuffled.select(range(N_METRICS))["abstract"]
medical_train_texts = pubmed_shuffled.select(range(N_METRICS, N_METRICS + N_TRAIN))["abstract"]
medical_heldout_texts = pubmed_shuffled.select(range(N_METRICS + N_TRAIN, N_METRICS + N_TRAIN + N_HELDOUT))["abstract"]

general_texts = [t for t in wikitext["text"] if t.strip()][:N_METRICS]

print(f"Medical metrics slice : {len(medical_metrics_texts)} abstracts")
print(f"Medical training slice: {len(medical_train_texts)} abstracts")
print(f"Medical held-out slice: {len(medical_heldout_texts)} abstracts")
print(f"General slice         : {len(general_texts)} non-empty WikiText lines")
print("\nExample held-out abstract:\n", medical_heldout_texts[0][:500])

## 2. Load a general-purpose and a medical tokenizer

**`gpt2`** is the archetypal modern tokenizer: byte-level BPE behind a single `tokenizer.json` file. The raw `tokenizers` library loads it directly — note that we use the low-level library itself here, not the `transformers` wrapper.

**`microsoft/biogpt`** is older and different in kind: a fairseq-style tokenizer = **Moses pre-tokenization + BPE with `</w>` end-of-word markers**, distributed as `vocab.json` + `merges.txt` (no `tokenizer.json`). We load it through `transformers.AutoTokenizer`, which is the reliable route for legacy repos like this one (recent transformers versions no longer ship a fast backend for it, and it needs the tiny `sacremoses` package).

The small `token_list()` adapter below lets us treat both uniformly — and is itself a good lesson in how many tokenizer flavors exist in the wild.

In [ ]:
from tokenizers import Tokenizer
from transformers import AutoTokenizer

tok_general = Tokenizer.from_pretrained("openai-community/gpt2")  # general-purpose, byte-level BPE
tok_medical = AutoTokenizer.from_pretrained("microsoft/biogpt")   # medical, Moses + fairseq-style BPE

def token_list(tok, text):
    """Tokens for `text` from either a raw `tokenizers.Tokenizer` or a transformers tokenizer."""
    if isinstance(tok, Tokenizer):
        return tok.encode(text, add_special_tokens=False).tokens
    return tok.tokenize(text)

print("gpt2   vocab size:", tok_general.get_vocab_size())
print("biogpt vocab size:", tok_medical.vocab_size)

### Same sentence, two tokenizers

Watch what happens to medical terms (drug names, eponyms, compound conditions). Special tokens are disabled so we compare *vocabularies*, not wrapper behavior.

In [ ]:
sentence = ("The patient was prescribed amoxicillin-clavulanate for "
            "community-acquired pneumonia and referred for echocardiography.")

for name, tok in [("gpt2 (general)", tok_general), ("biogpt (medical)", tok_medical)]:
    toks = token_list(tok, sentence)
    print(f"{name:17s} ({len(toks):2d} tokens): {toks}")

## 3. Quantify the mismatch: the 2×2 experiment

We measure, for each tokenizer on each corpus:

- **Fertility** — tokens per whitespace-separated word. Lower is better.
- **Tokens per 1k characters** — a context-window/compute view: how many positions 1000 characters of text consume.

If the medical tokenizer were simply "a better tokenizer", it would win everywhere. What we expect — and what proves the *domain-mismatch* point — is a **crossover**: each tokenizer wins on its home turf.

In [ ]:
def corpus_stats(tok, texts, tok_name, corpus_name):
    n_tokens = sum(len(token_list(tok, t)) for t in texts)
    n_words = sum(len(t.split()) for t in texts)
    n_chars = sum(len(t) for t in texts)
    return {
        "tokenizer": tok_name,
        "corpus": corpus_name,
        "fertility (tokens/word)": n_tokens / n_words,
        "tokens/1k chars": 1000.0 * n_tokens / n_chars,
    }

corpora = [("PubMed abstracts (medical)", medical_metrics_texts),
           ("WikiText-2 (general)", general_texts)]
tokenizer_list = [("gpt2", tok_general), ("biogpt", tok_medical)]

rows = [corpus_stats(tok, texts, tname, cname)
        for cname, texts in corpora for tname, tok in tokenizer_list]
df = pd.DataFrame(rows)
df.round(3)

In [ ]:
import numpy as np

x = np.arange(len(corpora))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 4.5))
for i, (tname, _) in enumerate(tokenizer_list):
    vals = [df[(df.tokenizer == tname) & (df.corpus == cname)]["fertility (tokens/word)"].iloc[0]
            for cname, _ in corpora]
    ax.bar(x + (i - 0.5) * width, vals, width, label=tname)
ax.set_xticks(x)
ax.set_xticklabels([c[0] for c in corpora])
ax.set_ylabel("fertility (tokens/word) — lower is better")
ax.set_title("Crossover effect: each tokenizer wins on its home domain")
ax.legend()
plt.show()

## 4. Check, don't assume: two famous "medical BERTs", two opposite choices

Being pretrained on medical text says nothing about the tokenizer. Two widely used biomedical encoders made **opposite vocabulary choices**:

- **BioClinicalBERT** (trained on MIMIC-III clinical notes) **reuses the stock `bert-base-cased` vocabulary** — "myocardial infarction" shatters into 8 pieces, exactly as generic BERT would do it.
- **PubMedBERT** (pretrained from scratch on PubMed) **retrained WordPiece on biomedical text** — "myocardial infarction" is two tokens, and ~18k of its 30,522 tokens don't exist in BERT's vocabulary at all.

Watch them tokenize the same sentence:

In [ ]:
bert_cased = AutoTokenizer.from_pretrained("bert-base-cased")
bio_clinical = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
pubmedbert = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext")

s = "The patient presented with acute myocardial infarction and was started on heparin."
for name, t in [("bert-base-cased", bert_cased),
                ("Bio_ClinicalBERT", bio_clinical),
                ("PubMedBERT", pubmedbert)]:
    print(f"{name:20s} vocab={t.vocab_size:6d}  {t.tokenize(s)}")

print("\nBio_ClinicalBERT vocabulary is exactly bert-base-cased's:",
      bio_clinical.get_vocab() == bert_cased.get_vocab())

SciBERT's *SciVocab* (~42% of its tokens do not appear in BERT's vocabulary at all) and BioGPT's PubMed-trained BPE are further examples of deliberately domain-specific vocabularies — but as the pair above shows, you cannot tell from the model card; you have to inspect the tokenizer files.

And the choice is genuinely debatable: PubMedBERT's own ablations (Gu et al., 2021) found that domain *vocabulary* alone gives much smaller downstream gains than domain *pretraining* — reusing BERT's vocab (the BioClinicalBERT choice) is a defensible shortcut. What a domain vocabulary unambiguously buys is **efficiency**: fewer tokens per word, longer effective context, cheaper training. That is exactly what we quantify in this notebook.

## 5. The fix: train our own medical tokenizer

Now the payoff. Using only the high-level `tokenizers` API we train a GPT-2-style byte-level BPE tokenizer **from scratch** on 5,000 PubMed abstracts:

1. **Model**: `BPE` — start with an empty merge table, learn merges from data.
2. **Pre-tokenizer**: `ByteLevel` — split on the GPT-2 regex and map every byte to a character in a 256-symbol alphabet, so there are **no unknown characters, ever**. We set `add_prefix_space=False` to match GPT-2's own behavior (no phantom leading space).
3. **Trainer**: `BpeTrainer` grows the vocabulary to 30,000 tokens, keeping pieces that occur at least twice. `initial_alphabet=ByteLevel.alphabet()` seeds all 256 byte-symbols first, so encoding → decoding stays lossless even for rare characters.

This runs on CPU in a couple of minutes — that is the whole point of the Rust backend.

In [ ]:
from tokenizers import pre_tokenizers
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

med_tokenizer = Tokenizer(BPE())
med_tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)  # match GPT-2's own setting
med_tokenizer.decoder = ByteLevelDecoder()     # must pair with the ByteLevel pre-tokenizer

trainer = BpeTrainer(
    vocab_size=30000,
    min_frequency=2,
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),  # all 256 bytes -> lossless decoding
    special_tokens=["<|endoftext|>"],
    show_progress=True,
)

med_tokenizer.train_from_iterator(medical_train_texts, trainer=trainer)
print("\nTrained vocab size:", med_tokenizer.get_vocab_size())

In [ ]:
# Save and reload — a trained tokenizer is a single portable JSON file.
med_tokenizer.save("medical_bpe_tokenizer.json")
reloaded = Tokenizer.from_file("medical_bpe_tokenizer.json")

enc = reloaded.encode(sentence, add_special_tokens=False)
print("tokens       :", enc.tokens)
print("round-trip   :", reloaded.decode(enc.ids))
print("matches orig :", sentence == reloaded.decode(enc.ids))

In [ ]:
# What did it learn? Whole-word medical pieces that gpt2 has to build from fragments.
vocab = med_tokenizer.get_vocab()
for term in ["myocardial", "pneumonia", "methotrexate"]:
    hits = sorted(t for t in vocab if term in t)
    print(f"{term:15s}: {hits[:6]}")

## 6. Head-to-head on held-out medical text

Final evaluation on the **held-out** PubMed slice — abstracts none of the tokenizers were trained on. We compare all three: the general-purpose `gpt2`, the professionally trained `biogpt`, and our from-scratch 5,000-abstract tokenizer.

In [ ]:
heldout_rows = [
    corpus_stats(tok, medical_heldout_texts, name, "held-out PubMed")
    for name, tok in [("gpt2 (general)", tok_general),
                      ("biogpt (medical)", tok_medical),
                      ("our trained BPE", reloaded)]
]
heldout_df = pd.DataFrame(heldout_rows)
heldout_df.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
names = heldout_df["tokenizer"]
vals = heldout_df["fertility (tokens/word)"]
bars = ax.bar(names, vals)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.3f}", ha="center", va="bottom")
ax.set_ylabel("fertility (tokens/word) — lower is better")
ax.set_title("Held-out PubMed abstracts: domain tokenizers win")
plt.tight_layout()
plt.show()

## 7. Wrap-up

**What we saw**

1. A general-purpose tokenizer (gpt2) needs substantially more tokens per word on PubMed abstracts than a medical tokenizer (biogpt) — and the effect **crosses over** on general text, so it is domain mismatch, not one tokenizer being "better".
2. "Medical model" doesn't imply "medical tokenizer": BioClinicalBERT reuses BERT's stock vocabulary (8 pieces for "myocardial infarction"), while PubMedBERT trained its own (2 pieces).
3. With the `tokenizers` library, a from-scratch byte-level BPE tokenizer trained on just 5,000 abstracts recovers most of the efficiency gap — in minutes, on CPU.

**Honest caveats for discussion**

- Lower fertility mainly buys **efficiency** (context length, training/inference cost). Downstream accuracy gains from vocabulary alone are modest next to domain pretraining (PubMedBERT ablations, Gu et al. 2021).
- We trained on 5k abstracts; a real deployment would use millions of documents and tune `vocab_size` against the target model size (Rust et al. find ~30k–100k a good trade-off).
- Tokenizer flavors differ (byte-level BPE, fairseq BPE with `</w>`, WordPiece with `[UNK]`), so always check *how* a tokenizer was built, not just its label. WordPiece tokenizers additionally need an unknown-token rate in this kind of analysis.

**Discussion questions for students**

1. Your model's context window is 4,096 tokens. Quantify how much more medical text fits with the domain tokenizer vs. gpt2 using the tokens/1k-chars numbers.
2. When would a *smaller* vocabulary be the right choice despite higher fertility?
3. Find a biomedical term that even biogpt splits. Why might that happen?

**Further reading**

- Rust et al., 2021 — *How Good is Your Tokenizer?* (fertility): https://arxiv.org/abs/2012.15613
- Beltagy et al., 2019 — SciBERT (domain vocabulary): https://arxiv.org/abs/1903.10676
- Gu et al., 2021 — PubMedBERT (vocab vs. pretraining ablations): https://arxiv.org/abs/2007.15779
- Hugging Face `tokenizers` quick tour: https://huggingface.co/docs/tokenizers/quicktour